# Synthetic Dataset Generation
# Part I a: Unstructured Reports


In [1]:
%pip install "pydantic>=2"


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from collections import defaultdict
from pathlib import Path
from random import uniform
from typing import Literal, Optional
import json
import random

from pydantic import BaseModel, ConfigDict, Field


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root whether Jupyter starts here or in the notebook folder."""
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        notebook = directory / "code_to_obtain_SR_data" / "synthetic_SR_dataset.ipynb"
        if notebook.exists():
            return directory
    raise FileNotFoundError("Could not locate the synthetic_dataset project root.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "code_to_obtain_SR_data" / "jsonl_data"
RAW_DATA_PATH = DATA_DIR / "synthetic_SR_100000.jsonl"
BALANCED_DATA_PATH = DATA_DIR / "balanced_synthetic_SR.jsonl"
DATA_DIR.mkdir(parents=True, exist_ok=True)

GENERATION_SEED = 42
BALANCING_SEED = 43


### Pydantic models to define the schema

**The generator returns plain dictionaries, but these models document and validate the generated JSONL structure.**


In [3]:
class LocalTumorStatus(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    morphology: Literal["polypoid", "semi-annular", "annular", "missing"] = Field(
        default="missing", alias='Morphology'
    )
    mucinous_component: Literal["yes", "no", "missing"] = Field(
        default="missing", alias='Mucinous component'
    )
    circumferential_tumor_involvement_start: Optional[int] = Field(
        ..., alias="Circumferential tumour involvement: from [ ] o'clock"
    )
    circumferential_tumor_involvement_end: Optional[int] = Field(
        ..., alias="Circumferential tumour involvement: to [ ] o'clock"
    )
    rectal_tumor_location_in_rectum: Literal["low", "mid", "high"] = Field(
        default="mid", alias='[low/mid/high] rectal tumour'
    )
    distance_from_anorectal_junction_to_distal_tumor_border_cm: Optional[float] = Field(
        ..., alias='Distance from anorectal junction to distal tumour border: [ ] cm'
    )
    relation_to_anterior_peritoneal_reflection: Literal["below", "at the level of", "above", "missing"] = Field(
        default="missing", alias='Relation to anterior peritoneal reflection'
    )
    tumor_length_cm: Optional[float] = Field(
        ..., alias='Tumour length: [ ] cm'
    )
    t_stage: Literal["T1_2", "T3a", "T3b", "T3c", "T3d", "T4a", "T4b", "missing"] = Field(
        default="missing", alias='T-stage'
    )
    invaded_structures: Optional[str] = Field(
        default="", alias='If T4b, structures with possible invasion: [ ]'  
    )
    sphincter_complex_invasion: Literal["no", "yes, internal sphincter only", "yes, internal sphincter and intersphincteric plane", "yes, internal sphincter and intersphincteric plane and external sphincter", "missing"] = Field(
        default="no", alias='Anal sphincter complex invasion'
    )
    lowest_part_of_sphincter_invasion: Literal["upper", "middle", "distal", "not applicable", "missing"] = Field(
        default="not applicable", alias='Lowest part of sphincter invasion'
    )


class MesorectalFasciaInvolement(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    shortest_distance_to_MRF_mm: Optional[float] = Field(
        ..., alias='Shortest distance to MRF: [ ] mm'  # not on chart
    )
    margin: Literal["clear(> 1 mm)", "involved(<= 1 mm)", "not applicable", "missing"] = Field(
        default="missing", alias='Mesorectal fascia (MRF) involvement'
    )
    circumferential_location_of_shortest_distance: Optional[int] = Field(
        ..., alias="If involved: at [ ] o'clock"
    )
    involved_by: Literal["tumour", "EMVI", "tumour deposit", "missing", "not applicable"] = Field(
        default="not applicable", alias='Involved by [tumour/EMVI/tumour deposit]'
    )


class LymphNodesAndTumourDeposits(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    regional_total_suspicious_mesorectal_nodes: Optional[int] = Field(
        ..., alias='Total number of suspicious mesorectal lymph nodes'
    )
    nodes_short_axis_ge_9mm: Optional[int] = Field(
        default=None, alias='# nodes with short axis ≥ 9 mm'
    )
    nodes_from_5_to_8mm_with_2_morphologic_criteria: Optional[int] = Field(
        default=None, alias='# nodes with short axis 5–8 mm and at least 2 morphologic suspicious criteria'
    )
    nodes_5mm_with_all_3_morphologic_criteria: Optional[int] = Field(
        default=None, alias='# nodes with short axis 5 mm and all 3 morphologic suspicious criteria'
    )
    mucinous_nodes_any_size: Optional[int] = Field(
        default=None, alias='# mucinous nodes (any size)'
    )
    regional_total_suspicious_extramesorectal_nodes: Optional[int] = Field(
        ..., alias='Total number of suspicious extramesorectal lymph nodes'
    )
    extramesorectal_location: Optional[str] = Field(
        default="", alias='Extramesorectal location'  # not on chart
    )
    n_stage: Literal["N0", "N1", "N2", "missing"] = Field(
        default="missing", alias='N-stage'
    )
    tumor_deposits_within_mesorectum: Optional[int] = Field(
        ..., alias='Number of tumour deposits within the mesorectum'
    )


class ExtramuralVascularInvasion(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    presence_of_emvi: Literal["yes", "no", "missing"] = Field(
        default="missing", alias='Presence of EMVI'
    )
    emvi_clock_position: Optional[int] = Field(
        ..., alias="If positive: at [ ] o'clock"
    )


class RectalCancerReport(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    local_tumor_status: LocalTumorStatus = Field(
        ..., alias='Local tumour status'
    )
    mesorectal_fascia_involement: MesorectalFasciaInvolement = Field(
        ..., alias='Mesorectal fascia (MRF) involvement'
    )
    lymph_nodes_and_tumor_deposits: LymphNodesAndTumourDeposits = Field(
        ..., alias='Lymph nodes and tumour deposits'
    )
    emvi: ExtramuralVascularInvasion = Field(
        ..., alias='Extramural vascular invasion (EMVI)'
    )


Check the rules again
Random sampling follows the intended distribution.


In [4]:
def angle_to_quadrant(angle: int) -> int:
    if 1 <= angle <= 3:
        return 1
    elif 4 <= angle <= 6:
        return 2
    elif 7 <= angle <= 9:
        return 3
    else:
        return 4

def get_quadrants_spanned(start: int, end: int, morphology: str = None) -> set[int]:
    if morphology == "annular":
        return {1, 2, 3, 4}
    quadrants = set()
    current = start
    while True:
        quadrants.add(angle_to_quadrant(current))
        if current == end:
            break
        current += 1
        if current > 12:
            current = 1
    return quadrants

def get_positions_spanned(start: int, end: int, morphology: str = None) -> list[int]:
    if morphology == "annular":
        return list(range(1, 13))
    positions = []
    current = start
    while True:
        positions.append(current)
        if current == end:
            break
        current = 1 if current == 12 else current + 1
    return positions


def calculate_tumour_length(t_stage: str) -> float:
    params = {
        "T1_2":(2, 1.0, 0.8, 3.5),
        "T3a":(2.2, 1.0, 1.2, 4.0),
        "T3b":(2.5, 1.2, 1.5, 5.5),
        "T3c":(3.5, 1.3, 2.0, 7.0),
        "T3d":(3.5, 1.5, 2.5, 8.0),
        "T4a":(4.5, 1.5, 2.0, 9.5),
        "T4b":(5, 1.5, 2.0, 10.0),
    }
    if t_stage not in params:
        raise ValueError(f"Unexpected t_stage for tumour length generation: {t_stage!r}")
    mean, sd, lo, hi = params[t_stage]
    x = random.gauss(mean, sd)
    x = max(lo, min(x, hi))
    return round(x, 1)

def sample_consistent_tumour_length(
    t_stage: str,
    distance_from_arj: float,
    apr_from_arj: float,
    target_relation: str | None = None,
    max_attempts: int = 200
) -> tuple[float, str]:
    for _ in range(max_attempts):
        x = calculate_tumour_length(t_stage)
        rel = relation_to_apr(distance_from_arj, x, apr_from_arj)
        if target_relation is None or rel == target_relation:
            return x, rel
    x = calculate_tumour_length(t_stage)
    return x, relation_to_apr(distance_from_arj, x, apr_from_arj)

def relation_to_apr(distance_from_arj: float, tumour_length_cm: float, apr_from_arj: float) -> str:
    # All measurements in cm from the anorectal junction.
    # inferior_edge = distance_from_arj (the distal tumour border)
    # superior_edge = distance_from_arj + tumour_length_cm
    inferior_edge = distance_from_arj
    superior_edge = distance_from_arj + max(tumour_length_cm, 0)
    if inferior_edge >= apr_from_arj:
        return "above"
    if superior_edge >= apr_from_arj:
        return "at the level of"
    return "below"

def location_by_bulk(distal_from_av, length_cm):
    """Classify rectal level by the sector holding MOST of the tumour (main burden),
    measured from the anorectal junction: low [0,5), mid [5,10), high [10, ...)."""
    if distal_from_av is None:
        return "low"
    sup = distal_from_av + max(length_cm or 0, 0)
    def ov(a, b):
        return max(0.0, min(sup, b) - max(distal_from_av, a))
    overlaps = {"low": ov(0, 5), "mid": ov(5, 10), "high": ov(10, 1e9)}
    return max(overlaps, key=lambda k: overlaps[k])

def sample_clock_span(min_steps: int = 1, max_steps: int = 3) -> tuple[int, int]:
    start = random.randint(1, 12)
    steps = random.randint(min_steps, max_steps)
    end = ((start - 1 + steps) % 12) + 1
    return start, end




QUADRANT_MAP = {
    1: {
        'above': [
            'small bowel',
            'large bowel',
            'urinary bladder',
            'ureter',
            'uterus',
            'cervix',
            'ovary'
        ],
        'at the level of': [
            'urinary bladder',
            'ureter',
            'seminal vesicle',
            'uterus',
            'cervix',
            'ovary'
        ],
        'below': [
            'urinary bladder',
            'urethra',
            'ureter',
            'prostate',
            'seminal vesicle',
            'vagina',
            'obturator muscle',
            'ischioanal space',
            'levator ani muscle',
            'puborectalis muscle'
        ]
    },
    2: {
        'above': [
            'small bowel',
            'large bowel',
            'internal iliac vessel',
            'piriformis muscle',
            'sacrum',
            'presacral fascia'
        ],
        'at the level of': [
            'sacrum',
            'presacral fascia',
            'internal iliac vessel'
        ],
        'below': [
            'obturator muscle',
            'ischioanal space',
            'sacrospinous ligament',
            'sacrotuberous ligament',
            'sacrum',
            'presacral fascia',
            'levator ani muscle',
            'puborectalis muscle'
        ]
    },
    3: {
        'above': [
            'small bowel',
            'large bowel',
            'internal iliac vessel',
            'piriformis muscle',
            'sacrum',
            'presacral fascia'
        ],
        'at the level of': [
            'sacrum',
            'presacral fascia',
            'internal iliac vessel'
        ],
        'below': [
            'obturator muscle',
            'ischioanal space',
            'sacrospinous ligament',
            'sacrotuberous ligament',
            'sacrum',
            'presacral fascia',
            'levator ani muscle',
            'puborectalis muscle'
        ]
    },
    4: {
        'above': [
            'small bowel',
            'large bowel',
            'urinary bladder',
            'ureter',
            'uterus',
            'cervix',
            'ovary'
        ],
        'at the level of': [
            'urinary bladder',
            'ureter',
            'seminal vesicle',
            'uterus',
            'cervix',
            'ovary'
        ],
        'below': [
            'urinary bladder',
            'urethra',
            'ureter',
            'prostate',
            'seminal vesicle',
            'vagina',
            'obturator muscle',
            'ischioanal space',
            'levator ani muscle',
            'puborectalis muscle'
        ]
    }
}

PAIRED_STRUCTURES = {
    "ureter",
    "ovary",
    "internal iliac vessel",
    "obturator muscle",
    "levator ani muscle",
    "puborectalis muscle",
    "piriformis muscle",
    "sacrospinous ligament",
    "sacrotuberous ligament",
    "ischioanal space",
    "seminal vesicle"
}


MIDLINE_STRUCTURES = {
    "urinary bladder",
    "vagina",
    "uterus",
    "cervix",
    "prostate",
    "sacrum",
    "presacral fascia",
    "rectosacral fascia",
    "peritoneal reflection",
    "large bowel",
    "sigmoid colon",
    "Denonvilliers' fascia",
    "rectovaginal septum"
}

QUADRANT_TO_SIDE = {
    1: "left",
    2: "left",
    3: "right",
    4: "right",
}

# Structures that require the distal tumour edge to be very close to the anal verge
VERY_LOW_ONLY_STRUCTURES = {
    'puborectalis muscle',
    'levator ani muscle',
    'ischioanal space'
}


BELOW_ONLY_T4B_STRUCTURES = {
    'urethra',
    'ischioanal space',
    'levator ani muscle',
    'puborectalis muscle',
    'piriformis muscle',
    'sacrospinous ligament',
    'sacrotuberous ligament'
}


def base_structure_name(invaded_structures: str) -> str:
    if not invaded_structures:
        return ""

    s = invaded_structures.lower().strip()
    for prefix in ("left ", "right ", "bilateral "):
        if s.startswith(prefix):
            s = s[len(prefix):]

    singular_map = {
        'ovaries': 'ovary',
        'internal iliac vessels': 'internal iliac vessel',
        'obturator muscles': 'obturator muscle',
        'levator ani muscles': 'levator ani muscle',
        'puborectalis muscles': 'puborectalis muscle',
        'piriformis muscles': 'piriformis muscle',
        'sacrospinous ligaments': 'sacrospinous ligament',
        'sacrotuberous ligaments': 'sacrotuberous ligament',
        'ischioanal spaces': 'ischioanal space',
        'seminal vesicles': 'seminal vesicle',
        'ureters': 'ureter',
    }
    return singular_map.get(s, s)

def assign_laterality(structure: str, quadrants_spanned: set) -> str:
    if structure in MIDLINE_STRUCTURES:
        return structure

    if structure in PAIRED_STRUCTURES:
        # Restrict to quadrants that are BOTH spanned AND actually contain this structure,
        # so a posterior structure is not called "right" from an anterior-right span.
        home_quadrants = {
            q for q in QUADRANT_MAP
            if any(structure in QUADRANT_MAP[q].get(rel, [])
                   for rel in ("above", "at the level of", "below"))
        }
        relevant_quadrants = (home_quadrants & set(quadrants_spanned)) or set(quadrants_spanned)
        sides = {QUADRANT_TO_SIDE[q] for q in relevant_quadrants if q in QUADRANT_TO_SIDE}

        bilateral_forms = {
            "ovary": "bilateral ovaries",
            "internal iliac vessel": "bilateral internal iliac vessels",
            "obturator muscle": "bilateral obturator muscles",
            "levator ani muscle": "bilateral levator ani muscles",
            "puborectalis muscle": "bilateral puborectalis muscles",
            "piriformis muscle": "bilateral piriformis muscles",
            "sacrospinous ligament": "bilateral sacrospinous ligaments",
            "sacrotuberous ligament": "bilateral sacrotuberous ligaments",
            "ischioanal space": "bilateral ischioanal spaces",
            "seminal vesicle": "bilateral seminal vesicles",
            "ureter": "bilateral ureters",
        }

        if sides == {"left", "right"}:
            return bilateral_forms.get(structure, f"bilateral {structure}s")
        if sides == {"left"}:
            return f"left {structure}"
        if sides == {"right"}:
            return f"right {structure}"

    return structure


def pick_invaded_structure_with_laterality(
    quadrants_spanned: set,
    apr_relation: str,
    tumour_location: str = "low",
    positions_spanned: list = None,
    distance_from_arj: float = None,
) -> str:
    def filtered_pool(pool):
        temp_pool = set(pool)

        if tumour_location != "low":
            temp_pool = {s for s in temp_pool if s not in VERY_LOW_ONLY_STRUCTURES}

        # EAS/levator/puborectalis/ischioanal only reachable when distal edge < 3.5 cm from AV
        if distance_from_arj is not None and distance_from_arj >= 3.5:
            temp_pool = {s for s in temp_pool if s not in VERY_LOW_ONLY_STRUCTURES}

        # For purely anterior or purely posterior spans, avoid strongly lateral targets.
        if positions_spanned:
            pos = set(positions_spanned)
            if pos.issubset({11, 12, 1, 5, 6, 7}):
                highly_lateral = {
                    'internal iliac vessel',
                    'obturator muscle',
                    'ischioanal space',
                    'piriformis muscle',
                    'sacrospinous ligament',
                    'sacrotuberous ligament',
                    'ureter',
                    'ovary'
                }
                temp_pool = {s for s in temp_pool if s not in highly_lateral}

        return temp_pool

    possible_targets = set()

    for q in quadrants_spanned:
        if q in QUADRANT_MAP:
            possible_targets.update(filtered_pool(QUADRANT_MAP[q].get(apr_relation, [])))

    if not possible_targets:
        if apr_relation == "below":
            fallback_order = ["at the level of", "above"]
        elif apr_relation == "at the level of":
            fallback_order = ["above", "below"]
        else:
            fallback_order = ["at the level of"]

        for fallback in fallback_order:
            for q in quadrants_spanned:
                if q in QUADRANT_MAP:
                    possible_targets.update(filtered_pool(QUADRANT_MAP[q].get(fallback, [])))
            if possible_targets:
                break

    if not possible_targets:
        possible_targets.update(filtered_pool(QUADRANT_MAP[4].get("above", [])))

    if not possible_targets:
        return "adjacent structure"

    structure = random.choice(sorted(possible_targets))
    return assign_laterality(structure, quadrants_spanned)

### Let's first generate structured reports where all the information is present. Then we'll randomly replace some fields as being "missing"

In [5]:
def generate_report():
    ### Local Tumour Status
    sphincter_complex_invasion = "no"
    lowest_part_of_sphincter_invasion = "not applicable"

    morphology_options = ["polypoid", "semi-annular", "annular"]
    morphology_weights = [1, 1, 0.25]
    morphology = random.choices(morphology_options, weights=morphology_weights)[0]
    mucinous_component = random.choices(["yes", "no"], weights=[0.4, 0.6])[0]

    if morphology == "annular":
        start = None
        end = None
    elif morphology == "polypoid":
        # Polypoid: <= 90° of circumference = 1–3 clock hours (1–2 steps)
        start, end = sample_clock_span(1, 2)
    else:
        # Semi-annular: > 90° and < 360° = 4–10 clock hours (3–9 steps)
        start, end = sample_clock_span(3, 9)

    # Length sampled up front, independent of T-stage (length is not part of T-staging).
    tumour_length_cm = round(uniform(2.0, 5.5), 1)

    # Anorectal junction (ARJ) is the landmark. Sample the distal-border distance from
    # the ARJ directly, uniform over 0.1-11.5 cm. 
    distance_from_arj = round(uniform(0.1, 11.5), 1)

    rectal_tumour_location_in_rectum = location_by_bulk(distance_from_arj, tumour_length_cm)

    apr_from_arj = round(max(7.0, min(11.0, random.gauss(9.0, 1.0))), 1)
    relation_to_anterior_peritoneal_reflection = relation_to_apr(
        distance_from_arj,
        tumour_length_cm,
        apr_from_arj
    )
    invaded_structures = ""
    apr_driven_t4b = False


    # Annular tumours span the full clock face and do not report start/end clocks.
    positions_spanned = get_positions_spanned(start, end, morphology)
    quadrants_spanned = get_quadrants_spanned(start, end, morphology)

    anterior_positions = {10, 11, 12, 1, 2}
    anterior_count = len(anterior_positions.intersection(positions_spanned))
    total_span = len(positions_spanned)

    apr_positive_high_rectum = (
        rectal_tumour_location_in_rectum == "high"
        and relation_to_anterior_peritoneal_reflection in ["above", "at the level of"]
        and anterior_count >= 2
        and total_span <= 7
    )
    apr_positive_mid_rectum = (
        rectal_tumour_location_in_rectum == "mid"
        and relation_to_anterior_peritoneal_reflection == "at the level of"
        and anterior_count >= 2
        and total_span <= 6
    )
    if apr_positive_high_rectum or apr_positive_mid_rectum:
        t_stage = random.choices(["T4a", "T4b"], weights=[0.7, 0.3])[0]
        if t_stage == "T4a":
            if relation_to_anterior_peritoneal_reflection == "above":
                invaded_structures = "peritoneum"
            else:
                invaded_structures = "peritoneal reflection"
        else:
            apr_driven_t4b = True
            invaded_structures = ""
        sphincter_complex_invasion = "no"
        lowest_part_of_sphincter_invasion = "not applicable"

    elif rectal_tumour_location_in_rectum == "low":
        if distance_from_arj > 1.0:
            t_options = ["T1_2", "T3a", "T3b", "T3c", "T3d", "T4b"]
            t_weights = [0.8, 1.0, 1.5, 1.5, 2.0, 2.0] if morphology == "polypoid" else [0.5, 1.0, 1.5, 1.5, 2.0, 2.0]
            t_stage = random.choices(t_options, weights=t_weights)[0]
            invaded_structures = (
                pick_invaded_structure_with_laterality(
                    quadrants_spanned,
                    relation_to_anterior_peritoneal_reflection,
                    tumour_location="low",
                    positions_spanned=positions_spanned,
                    distance_from_arj=distance_from_arj,
                )
                if t_stage == "T4b"
                else ""
            )
        else:
            sph_options = [
                "no",
                "yes, internal sphincter only",
                "yes, internal sphincter and intersphincteric plane",
                "yes, internal sphincter and intersphincteric plane and external sphincter",
            ]
            sph_weights = [0.15, 0.15, 0.25, 0.35]
            sphincter_complex_invasion = random.choices(sph_options, weights=sph_weights)[0]
            if sphincter_complex_invasion != "no":
                lowest_part_of_sphincter_invasion = random.choice(["distal", "middle"])

            if sphincter_complex_invasion.endswith("external sphincter"):
                t_stage = "T4b"
            else:
                t_options = ["T1_2", "T3a", "T3b", "T3c", "T3d", "T4b"]
                t_weights = [0.8, 1.0, 2.0, 2.0, 3.0, 2.5] if morphology == "polypoid" else [0.5, 1.0, 2.0, 2.0, 3.0, 3.0]
                t_stage = random.choices(t_options, weights=t_weights)[0]

            if sphincter_complex_invasion == "yes, internal sphincter only":
                t_stage = "T1_2"
            if t_stage == "T1_2" and "intersphincteric plane" in sphincter_complex_invasion:
                t_stage = "T3a"

            if t_stage == "T4b":
                if sphincter_complex_invasion.endswith("external sphincter"):
                    pelvic_structure = random.choices(
                        ["levator ani muscle", "puborectalis muscle", "ischioanal space"],
                        weights=[1, 1, 0.4],
                    )[0]
                    invaded_structures = assign_laterality(
                        pelvic_structure, quadrants_spanned
                    )
                else:
                    invaded_structures = pick_invaded_structure_with_laterality(
                        quadrants_spanned,
                        "below",
                        tumour_location="low",
                        positions_spanned=positions_spanned,
                        distance_from_arj=distance_from_arj,
                    )
            else:
                invaded_structures = ""
    else:
        # T4a requires tumour reaching APR: only include when relation is above or at the level of
        if relation_to_anterior_peritoneal_reflection in ["above", "at the level of"]:
            t_options = ["T1_2", "T3a", "T3b", "T3c", "T3d", "T4a", "T4b"]
            t_weights = [0.7, 1, 1, 1, 1, 0.5, 1.5] if morphology == "polypoid" else [0.5, 1, 1, 1, 1, 1.0, 1.5]
        else:
            t_options = ["T1_2", "T3a", "T3b", "T3c", "T3d", "T4b"]
            t_weights = [0.7, 1, 1, 1, 1, 1.5] if morphology == "polypoid" else [0.5, 1, 1, 1, 1, 1.5]
        t_stage = random.choices(t_options, weights=t_weights)[0]
        sphincter_complex_invasion = "no"
        lowest_part_of_sphincter_invasion = "not applicable"

        if t_stage == "T4b":
            invaded_structures = pick_invaded_structure_with_laterality(
                quadrants_spanned, relation_to_anterior_peritoneal_reflection,
                tumour_location=rectal_tumour_location_in_rectum, positions_spanned=positions_spanned, distance_from_arj=distance_from_arj
            )
        elif t_stage == "T4a":
            if relation_to_anterior_peritoneal_reflection == "above":
                invaded_structures = "peritoneum"
            else:
                invaded_structures = "peritoneal reflection"
        else:
            invaded_structures = ""


    if t_stage == "T1_2":
        mucinous_component = random.choices(["yes", "no"], weights=[0.03, 0.97])[0]
        if morphology == "annular":
            morphology = random.choices(["polypoid", "semi-annular"], weights=[0.75, 0.25])[0]
            if morphology == "polypoid":
                start, end = sample_clock_span(1, 2)
            else:
                start, end = sample_clock_span(3, 5)
            positions_spanned = get_positions_spanned(start, end, morphology)
            quadrants_spanned = get_quadrants_spanned(start, end, morphology)

    target_relation = None
    if t_stage == "T4a":
        target_relation = "above" if invaded_structures == "peritoneum" else "at the level of"
    elif apr_driven_t4b:
        target_relation = relation_to_anterior_peritoneal_reflection
    elif t_stage == "T4b" and rectal_tumour_location_in_rectum == "low":
        if base_structure_name(invaded_structures) in BELOW_ONLY_T4B_STRUCTURES:
            target_relation = "below"

    relation_to_anterior_peritoneal_reflection = relation_to_apr(
        distance_from_arj, tumour_length_cm, apr_from_arj
    )
    if target_relation is not None and relation_to_anterior_peritoneal_reflection != target_relation:
        for _cand_apr in (round(uniform(7.0, 11.0), 1) for _ in range(200)):
            if relation_to_apr(distance_from_arj, tumour_length_cm, _cand_apr) == target_relation:
                apr_from_arj = _cand_apr
                relation_to_anterior_peritoneal_reflection = target_relation
                break

    if (
        t_stage == "T4b"
        and rectal_tumour_location_in_rectum == "low"
        and base_structure_name(invaded_structures) in BELOW_ONLY_T4B_STRUCTURES
        and relation_to_anterior_peritoneal_reflection != "below"
    ):
        tumour_length_cm, relation_to_anterior_peritoneal_reflection = sample_consistent_tumour_length(
            t_stage,
            distance_from_arj,
            apr_from_arj,
            target_relation="below"
        )

    if t_stage == "T4a":
        if relation_to_anterior_peritoneal_reflection == "above":
            invaded_structures = "peritoneum"
        elif relation_to_anterior_peritoneal_reflection == "at the level of":
            invaded_structures = "peritoneal reflection"
        else:
            # Attempt to find a tumour length that reaches the APR
            tumour_length_cm, relation_to_anterior_peritoneal_reflection = sample_consistent_tumour_length(
                t_stage,
                distance_from_arj,
                apr_from_arj,
                target_relation="at the level of"
            )
            if relation_to_anterior_peritoneal_reflection in ["above", "at the level of"]:
                invaded_structures = "peritoneum" if relation_to_anterior_peritoneal_reflection == "above" else "peritoneal reflection"
            else:
                # Cannot reach APR: downgrade to T3d
                t_stage = "T3d"
                invaded_structures = ""
    elif t_stage == "T4b":
        if apr_driven_t4b and relation_to_anterior_peritoneal_reflection != "below":
            if relation_to_anterior_peritoneal_reflection == "at the level of":
                ANTERIOR_APR_T4B_STRUCTURES = [
                    'urinary bladder', 'ureter', 'uterus', 'cervix', 'ovary', 'seminal vesicle'
                ]
            else:  
                ANTERIOR_APR_T4B_STRUCTURES = [
                    'urinary bladder', 'ureter', 'uterus', 'cervix', 'ovary'
                ]
            structure = random.choice(ANTERIOR_APR_T4B_STRUCTURES)
            invaded_structures = assign_laterality(structure, quadrants_spanned)
        elif sphincter_complex_invasion == "yes, internal sphincter and intersphincteric plane and external sphincter" and rectal_tumour_location_in_rectum == "low":
            # EAS invasion is captured in sphincter_complex_invasion field;
            # invaded_structures reports additional pelvic floor structure
            pelvic_structure = random.choices(
                ["levator ani muscle", "puborectalis muscle", "ischioanal space"],
                weights=[1, 1, 0.4]
            )[0]
            invaded_structures = assign_laterality(pelvic_structure, quadrants_spanned)
        elif invaded_structures == "" or invaded_structures is None:
            invaded_structures = pick_invaded_structure_with_laterality(
                quadrants_spanned, relation_to_anterior_peritoneal_reflection,
                tumour_location=rectal_tumour_location_in_rectum, positions_spanned=positions_spanned, distance_from_arj=distance_from_arj
            )
    # FIX: Escalate sphincter invasion when T4b invades pelvic floor structures
    if t_stage == "T4b" and rectal_tumour_location_in_rectum == "low" and distance_from_arj is not None and distance_from_arj <= 1.0:
        # If invaded structure implies full sphincter breach, escalate
        pelvic_floor_structures = {'external anal sphincter', 'levator ani muscle', 'puborectalis muscle', 'ischioanal space'}
        # Check all structures in case of comma-separated list
        inv_parts = [base_structure_name(s.strip()) for s in invaded_structures.split(',')] if invaded_structures else []
        if any(p in pelvic_floor_structures for p in inv_parts):
            if sphincter_complex_invasion != "yes, internal sphincter and intersphincteric plane and external sphincter":
                sphincter_complex_invasion = "yes, internal sphincter and intersphincteric plane and external sphincter"
                if distance_from_arj is not None and distance_from_arj <= 2.5:
                    lowest_part_of_sphincter_invasion = random.choice(["distal", "middle"])
                else:
                    lowest_part_of_sphincter_invasion = random.choice(["middle", "upper"])

    local_status = {
        "morphology": morphology,
        "mucinous_component": mucinous_component,
        "circumferential_tumor_involvement_start": start,
        "circumferential_tumor_involvement_end": end,
        "rectal_tumor_location_in_rectum": rectal_tumour_location_in_rectum,
        "distance_from_anorectal_junction_to_distal_tumor_border_cm": distance_from_arj,
        "relation_to_anterior_peritoneal_reflection": relation_to_anterior_peritoneal_reflection,
        "tumor_length_cm": tumour_length_cm,
        "t_stage": t_stage,
        "invaded_structures": invaded_structures,
        "sphincter_complex_invasion": sphincter_complex_invasion,
        "lowest_part_of_sphincter_invasion": lowest_part_of_sphincter_invasion
    }

    ### Mesorectal Fascia Involvement
    mrf_distance = None
    margin = "clear(> 1 mm)"
    mrf_clock_position = None
    structure = "not applicable"

    # Peritoneum covers the anterior wall (no MRF there): 9-3 o'clock above the
    # reflection, 10-2 at the level of it.
    if relation_to_anterior_peritoneal_reflection == "above":
        anterior_apr_positions = {9, 10, 11, 12, 1, 2, 3}
    else:
        anterior_apr_positions = {10, 11, 12, 1, 2}
    if relation_to_anterior_peritoneal_reflection in ["above", "at the level of"]:
        valid_mrf_positions = [x for x in positions_spanned if x not in anterior_apr_positions]
    else:
        valid_mrf_positions = list(positions_spanned)

    apr_mrf_positive = (
        relation_to_anterior_peritoneal_reflection in ["above", "at the level of"]
        and bool(valid_mrf_positions)
    )

    t3_low_short_exception = (
        t_stage in ["T3a", "T3b", "T3c", "T3d"]
        and distance_from_arj is not None
        and distance_from_arj <= 2
        and tumour_length_cm is not None
        and tumour_length_cm < 2.2
    )
    # When tumour is entirely within the anterior APR zone (10-2 o'clock) and
    # above/at the level of the peritoneal reflection, there is no MRF anteriorly 
    # the anterior wall is covered by peritoneum, not mesorectal fascia.
    # This applies to all T-stages, not just T4a.
    anterior_no_mrf = (
        relation_to_anterior_peritoneal_reflection in ["above", "at the level of"]
        and bool(positions_spanned)
        and set(positions_spanned).issubset(anterior_apr_positions)
    )
    # If the tumour's superior edge barely reaches above the anal canal (< 4 cm from AV),
    # the mesorectum is too thin for meaningful MRF assessment.
    superior_edge_cm = (distance_from_arj or 0) + (tumour_length_cm or 0)
    tumour_below_mesorectum = superior_edge_cm < 4.0

    mrf_not_applicable = (
        t_stage == "T1_2"
        or t3_low_short_exception
        or anterior_no_mrf
        or tumour_below_mesorectum
    )

    if mrf_not_applicable:
        mrf_distance = None
        margin = "not applicable"
        mrf_clock_position = None
        structure = "not applicable"
    elif t_stage == "T4a":
        if apr_mrf_positive:
            mrf_distance = round(uniform(0.1, 5.0), 1)
            margin = "clear(> 1 mm)" if mrf_distance > 1 else "involved(<= 1 mm)"
            if margin == "involved(<= 1 mm)":
                mrf_clock_position = random.choice(valid_mrf_positions) if valid_mrf_positions else None
                structure = random.choices(["tumour", "EMVI", "tumour deposit"], weights=[1, 0.2, 0.2])[0]
        else:
            mrf_distance = round(uniform(1.1, 5.0), 1)
    elif t_stage == "T4b":
        valid_pos = list(valid_mrf_positions) if valid_mrf_positions else []
        mrf_distance = 0
        margin = "involved(<= 1 mm)"
        structure = random.choices(["tumour", "EMVI", "tumour deposit"], weights=[1, 0.2, 0.2])[0]

        if structure == "tumour":
            # Direct tumour invasion: MRF clock must be within the tumour's circumferential span
            mrf_clock_position = random.choice(valid_pos) if valid_pos else (
                random.choice(list(positions_spanned)) if positions_spanned else None
            )
        else:
            # EMVI / tumour deposit can track through mesorectum away from the primary
            if sphincter_complex_invasion == "yes, internal sphincter and intersphincteric plane and external sphincter" and rectal_tumour_location_in_rectum == "low":
                preferred_positions = [x for x in valid_pos if x not in anterior_apr_positions]
                fallback_positions = preferred_positions if preferred_positions else list(positions_spanned)
                mrf_clock_position = random.choice(fallback_positions)
            else:
                mrf_clock_position = random.choice(valid_pos) if valid_pos else None
    elif t_stage in ["T3a", "T3b", "T3c", "T3d"]:
        mrf_distance = round(uniform(0.5, 8.0), 1) if t_stage in ["T3a", "T3b"] else round(uniform(0.2, 3.0), 1)
        margin = "involved(<= 1 mm)" if mrf_distance <= 1 else "clear(> 1 mm)"
        if margin == "involved(<= 1 mm)":
            structure = random.choices(["tumour", "EMVI", "tumour deposit"], weights=[1, 0.2, 0.2])[0]
            if structure == "tumour":
                mrf_clock_position = random.choice(valid_mrf_positions) if valid_mrf_positions else (
                    random.choice(list(positions_spanned)) if positions_spanned else None
                )
            else:
                mrf_clock_position = random.choice(valid_mrf_positions) if valid_mrf_positions else None

    mesorectal = {
        "shortest_distance_to_MRF_mm": mrf_distance,
        "margin": margin,
        "circumferential_location_of_shortest_distance": mrf_clock_position,
        "involved_by": structure
    }

    ### Lymph Nodes and Tumour Deposits
    base_prob_suspicious = {"T1_2": 0.20, "T3a": 0.30, "T3b": 0.30, "T3c": 0.65, "T3d": 0.65, "T4a": 0.70}.get(t_stage, 0.80)
    # Cap maximum meso nodes at 6, to honour the maximum limit rule
    max_meso_nodes = {"T1_2": 2, "T3a": 3, "T3b": 3, "T3c": 4, "T3d": 5, "T4a": 6}.get(t_stage, 6)

    if tumour_length_cm is not None and tumour_length_cm > 5: base_prob_suspicious += 0.10
    elif tumour_length_cm is not None and tumour_length_cm < 2: base_prob_suspicious -= 0.10
    if mucinous_component == "yes": base_prob_suspicious += 0.05
    base_prob_suspicious = max(0.05, min(0.90, base_prob_suspicious))

    total_suspicious_mesorectal = random.randint(1, max_meso_nodes) if random.random() < base_prob_suspicious else 0
    meso_nodes_ge_9mm = meso_nodes_5_8mm_2criteria = meso_nodes_lt_5mm_3criteria = meso_mucinous_nodes = 0

    if total_suspicious_mesorectal > 0:
        remaining = total_suspicious_mesorectal
        if mucinous_component == "yes" and random.random() < 0.40:
            meso_mucinous_nodes = random.randint(1, min(remaining, 3))
            remaining -= meso_mucinous_nodes

        while remaining > 0:
            node_type = random.choices(
                ["gt_9", "5_8", "lt_5"],
                weights=[0.35, 0.60, 0.05]
            )[0]

            if node_type == "gt_9":
                meso_nodes_ge_9mm += 1
            elif node_type == "5_8":
                meso_nodes_5_8mm_2criteria += 1
            else:
                meso_nodes_lt_5mm_3criteria += 1
            remaining -= 1

    extrameso_prob = {"T3b": 0.10, "T3c": 0.25, "T3d": 0.25, "T4a": 0.30, "T4b": 0.40}.get(t_stage, 0.00)
    if rectal_tumour_location_in_rectum == "high":
        extrameso_prob = 0.02

    # Extramesorectal (obturator / internal iliac) nodes are suspicious by size only (>= 7 mm);
    # morphologic criteria and a size sub-breakdown do not apply to these nodes.
    total_suspicious_lateral = 0
    extrameso_location = ""

    if random.random() < extrameso_prob:
        # Enforce hard cap limit of 6 total regional nodes
        max_lateral_allowed = 6 - total_suspicious_mesorectal
        if max_lateral_allowed > 0:
            total_suspicious_lateral = random.randint(1, min(4, max_lateral_allowed))

            # Assign laterality from actual sided tumour extent; reserve "bilateral" for true left+right spread.
            node_basin = random.choice(["obturator", "internal iliac"])
            node_sides = {QUADRANT_TO_SIDE[q] for q in quadrants_spanned if q in QUADRANT_TO_SIDE}
            if node_sides == {"left", "right"}:
                node_laterality = "bilateral"
            elif node_sides == {"left"}:
                node_laterality = "left"
            elif node_sides == {"right"}:
                node_laterality = "right"
            else:
                node_laterality = random.choice(["left", "right"])

            # Bilateral only valid when >= 2 nodes; downgrade single node to one side
            if node_laterality == "bilateral" and total_suspicious_lateral < 2:
                node_laterality = random.choice(["left", "right"])

            if node_laterality == "bilateral":
                extrameso_location = f"{node_laterality} {node_basin} lymph nodes"
            else:
                extrameso_location = f"{node_laterality} {node_basin} lymph node"

    total_regional = total_suspicious_mesorectal + total_suspicious_lateral
    # Stage mapping: N0 = 0 nodes, N1 = 1-3 nodes, N2 = 4-6 nodes
    n_stage = "N0" if total_regional == 0 else ("N1" if total_regional <= 3 else "N2")

    dep_prob = {"T1_2": 0.08, "T3a": 0.12, "T3b": 0.18, "T3c": 0.22, "T3d": 0.25, "T4a": 0.20, "T4b": 0.25}.get(t_stage, 0.20)
    max_deposits = 2 if t_stage == "T1_2" else 3
    tumour_deposits = random.randint(1, max_deposits) if structure == "tumour deposit" or random.random() < dep_prob else 0
    if tumour_deposits > 0 and n_stage == "N0": n_stage = "N1"

    lymph_node_data = {
        "regional_total_suspicious_mesorectal_nodes": total_suspicious_mesorectal,
        "nodes_short_axis_ge_9mm": meso_nodes_ge_9mm,
        "nodes_from_5_to_8mm_with_2_morphologic_criteria": meso_nodes_5_8mm_2criteria,
        "nodes_5mm_with_all_3_morphologic_criteria": meso_nodes_lt_5mm_3criteria,
        "mucinous_nodes_any_size": meso_mucinous_nodes,
        "regional_total_suspicious_extramesorectal_nodes": total_suspicious_lateral,
        "extramesorectal_location": extrameso_location if total_suspicious_lateral > 0 else "",
        "n_stage": n_stage,
        "tumor_deposits_within_mesorectum": tumour_deposits
    }

    ### Extramural Vascular Invasion (EMVI)
    emvi_present = False
    emvi_clock_position = None
    if structure == "EMVI":
        emvi_present = True
        # EMVI clock must lie within the tumour's circumferential span. When the MRF is
        # involved by EMVI the MRF and EMVI clocks coincide, so pin both to one in-span
        # position (updating the MRF entry if its clock had fallen outside the span).
        if mrf_clock_position is not None and mrf_clock_position in positions_spanned:
            emvi_clock_position = mrf_clock_position
        else:
            emvi_clock_position = random.choice(positions_spanned) if positions_spanned else None
            mrf_clock_position = emvi_clock_position
            mesorectal["circumferential_location_of_shortest_distance"] = emvi_clock_position
    elif t_stage != "T1_2":
        emvi_prob = {"T3a": 0.05, "T3b": 0.15, "T3c": 0.35, "T3d": 0.50, "T4a": 0.60, "T4b": 0.65}.get(t_stage, 0)
        if random.random() < emvi_prob:
            emvi_present = True
            emvi_clock_position = random.choice(positions_spanned)

    full_report = {
        "local_tumor_status": local_status,
        "mesorectal_fascia_involement": mesorectal,
        "lymph_nodes_and_tumor_deposits": lymph_node_data,
        "emvi": {
            "presence_of_emvi": "yes" if emvi_present else "no",
            "emvi_clock_position": emvi_clock_position
        }
    }
    return full_report

random.seed(GENERATION_SEED)
with RAW_DATA_PATH.open("w", encoding="utf-8") as f:
    for _ in range(100_000):
        report = generate_report()
        RectalCancerReport.model_validate(report)
        f.write(json.dumps(report) + "\n")


### We generated 100000 examples in order to create a smaller balanced dataset with 1000 examples

**We will check the distributions of values within each field**

In [6]:
def get_value_distributions(outputs_iterator):
    distributions = defaultdict(lambda: defaultdict(int))

    def collect_values(data, path=""):
        if isinstance(data, dict):
            for key, value in data.items():
                new_path = f"{path}.{key}" if path else key
                collect_values(value, new_path)
        elif isinstance(data, list):
            distributions[path][str(data)] += 1
        else:
            distributions[path][str(data)] += 1

    for parsed_output in outputs_iterator:
        collect_values(parsed_output)
    return distributions


def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} on line {line_num}") from exc
    return records


all_outputs = load_jsonl(RAW_DATA_PATH)
print(f"Successfully loaded {len(all_outputs)} records from {RAW_DATA_PATH}")

distributions = get_value_distributions(all_outputs)
print("\n" + "=" * 50)
print("Value Distributions Across All Fields:")
print("=" * 50 + "\n")
for field_path, value_counts in sorted(distributions.items()):
    print(f"Field: {field_path}")
    total_occurrences = sum(value_counts.values())
    for value, count in value_counts.items():
        percentage = (count / total_occurrences) * 100
        print(f"  '{value}': {count} occurrences ({percentage:.2f}%)")
    print("\n")
    print("-" * 67)


Successfully loaded 100000 records from C:\Users\simmu\Desktop\Paper\synthetic_dataset\code_to_obtain_SR_data\jsonl_data\synthetic_SR_100000.jsonl

Value Distributions Across All Fields:

Field: emvi.emvi_clock_position
  'None': 57699 occurrences (57.70%)
  '12': 3864 occurrences (3.86%)
  '7': 3345 occurrences (3.35%)
  '5': 3295 occurrences (3.29%)
  '6': 3312 occurrences (3.31%)
  '2': 3453 occurrences (3.45%)
  '1': 3758 occurrences (3.76%)
  '10': 3482 occurrences (3.48%)
  '9': 3526 occurrences (3.53%)
  '11': 3823 occurrences (3.82%)
  '8': 3394 occurrences (3.39%)
  '3': 3550 occurrences (3.55%)
  '4': 3499 occurrences (3.50%)


-------------------------------------------------------------------
Field: emvi.presence_of_emvi
  'no': 57699 occurrences (57.70%)
  'yes': 42301 occurrences (42.30%)


-------------------------------------------------------------------
Field: local_tumor_status.circumferential_tumor_involvement_end
  '9': 7270 occurrences (7.27%)
  '6': 7475 occurren

 ### We will balance the dataset with t-stage and tumour location in rectum in order to have all the variants present in the final dataset.

In [7]:
# Reuse the records already loaded above instead of keeping a second 100,000-record copy.
raw_records_full = all_outputs
final_dataset_size = 1_000

# Explicit joint targets enforce the domain rule that T4a cannot occur in the
# low rectal segment. Row totals preserve the normalized relative T-stage
# weights; column totals remain high=300, mid=200, low=500.
combined_target_counts = {
    ("T1_2", "high"): 24, ("T1_2", "mid"): 16, ("T1_2", "low"): 71,
    ("T3a", "high"): 12, ("T3a", "mid"): 8, ("T3a", "low"): 36,
    ("T3b", "high"): 24, ("T3b", "mid"): 16, ("T3b", "low"): 71,
    ("T3c", "high"): 12, ("T3c", "mid"): 8, ("T3c", "low"): 36,
    ("T3d", "high"): 24, ("T3d", "mid"): 16, ("T3d", "low"): 71,
    ("T4a", "high"): 133, ("T4a", "mid"): 89, ("T4a", "low"): 0,
    ("T4b", "high"): 71, ("T4b", "mid"): 47, ("T4b", "low"): 215,
}
assert sum(combined_target_counts.values()) == final_dataset_size

combined_groups = defaultdict(list)
for record in raw_records_full:
    local_status = record["local_tumor_status"]
    key = (
        local_status["t_stage"],
        local_status["rectal_tumor_location_in_rectum"],
    )
    combined_groups[key].append(record)

shortages = {
    key: target_count - len(combined_groups[key])
    for key, target_count in combined_target_counts.items()
    if len(combined_groups[key]) < target_count
}
if shortages:
    raise ValueError(
        "The generated pool cannot satisfy the requested joint balance: "
        f"{shortages}"
    )

random.seed(BALANCING_SEED)
balanced_dataset = []
for key, target_count in combined_target_counts.items():
    balanced_dataset.extend(random.sample(combined_groups[key], target_count))

random.shuffle(balanced_dataset)
assert len(balanced_dataset) == final_dataset_size
print(f"Successfully created a balanced dataset with {len(balanced_dataset)} records.")


Successfully created a balanced dataset with 1000 records.


**Saving a balanced dataset**

In [8]:
with BALANCED_DATA_PATH.open("w", encoding="utf-8") as f:
    for record in balanced_dataset:
        f.write(json.dumps(record) + "\n")
print(f"Successfully saved {len(balanced_dataset)} records to '{BALANCED_DATA_PATH}'.")


Successfully saved 1000 records to 'C:\Users\simmu\Desktop\Paper\synthetic_dataset\code_to_obtain_SR_data\jsonl_data\balanced_synthetic_SR.jsonl'.


In [9]:
# Analyse the dataset just created; do not reload a stale file from a prior run.
balanced_distributions = get_value_distributions(balanced_dataset)
print("\n" + "=" * 50)
print("Value Distributions Across Balanced Dataset Fields:")
print("=" * 50 + "\n")
for field_path, value_counts in sorted(balanced_distributions.items()):
    print(f"Field: {field_path}")
    total_occurrences = sum(value_counts.values())
    for value, count in value_counts.items():
        percentage = (count / total_occurrences) * 100
        print(f"  '{value}': {count} occurrences ({percentage:.2f}%)")
    print("\n")
    print("-" * 67)



Value Distributions Across Balanced Dataset Fields:

Field: emvi.emvi_clock_position
  'None': 538 occurrences (53.80%)
  '2': 34 occurrences (3.40%)
  '3': 38 occurrences (3.80%)
  '8': 32 occurrences (3.20%)
  '10': 46 occurrences (4.60%)
  '11': 51 occurrences (5.10%)
  '7': 30 occurrences (3.00%)
  '5': 38 occurrences (3.80%)
  '4': 44 occurrences (4.40%)
  '9': 23 occurrences (2.30%)
  '6': 39 occurrences (3.90%)
  '1': 48 occurrences (4.80%)
  '12': 39 occurrences (3.90%)


-------------------------------------------------------------------
Field: emvi.presence_of_emvi
  'no': 538 occurrences (53.80%)
  'yes': 462 occurrences (46.20%)


-------------------------------------------------------------------
Field: local_tumor_status.circumferential_tumor_involvement_end
  '7': 63 occurrences (6.30%)
  '5': 84 occurrences (8.40%)
  '2': 77 occurrences (7.70%)
  'None': 98 occurrences (9.80%)
  '3': 79 occurrences (7.90%)
  '9': 84 occurrences (8.40%)
  '11': 77 occurrences (7.70%)
  

In [10]:
for i, report in enumerate(balanced_dataset):
    if i >= 30:
        break
    print(json.dumps(report, indent=2))
    print("\n" + "-" * 80 + "\n")

{
  "local_tumor_status": {
    "morphology": "polypoid",
    "mucinous_component": "no",
    "circumferential_tumor_involvement_start": 5,
    "circumferential_tumor_involvement_end": 7,
    "rectal_tumor_location_in_rectum": "low",
    "distance_from_anorectal_junction_to_distal_tumor_border_cm": 1.1,
    "relation_to_anterior_peritoneal_reflection": "below",
    "tumor_length_cm": 5.1,
    "t_stage": "T3b",
    "invaded_structures": "",
    "sphincter_complex_invasion": "no",
    "lowest_part_of_sphincter_invasion": "not applicable"
  },
  "mesorectal_fascia_involement": {
    "shortest_distance_to_MRF_mm": 6.2,
    "margin": "clear(> 1 mm)",
    "circumferential_location_of_shortest_distance": null,
    "involved_by": "not applicable"
  },
  "lymph_nodes_and_tumor_deposits": {
    "regional_total_suspicious_mesorectal_nodes": 1,
    "nodes_short_axis_ge_9mm": 0,
    "nodes_from_5_to_8mm_with_2_morphologic_criteria": 1,
    "nodes_5mm_with_all_3_morphologic_criteria": 0,
    "mucino